
# V44 LIVE — Sexual-Content Specialist

**Protected public-LB champion:** V38 = **0.608**  
**Target:** **0.620**

V42 sarcasm failed its validation gate.  
V43 identity attack also failed its validation gate.

V44 therefore targets the last meaningful untouched label: `sexual_content`.

The LIVE training set contains only **159 sexual-content positives**, so V44 deliberately combines two very different models:

1. **NB-SVM lexical expert**
   - word TF-IDF
   - character TF-IDF
   - Naive-Bayes log-count ratios
   - logistic classifier
   - especially useful for explicit words, misspellings, spacing tricks and Banglish

2. **Twitter-XLM-R sexual-content expert**
   - `cardiffnlp/twitter-xlm-roberta-base`
   - dedicated binary fine-tuning
   - social-media pretraining
   - hard-negative emphasis on profanity-positive but sexual-negative text
   - FGM adversarial training
   - pairwise ranking loss
   - layer-wise learning-rate decay

## Generalization controls

- exact accepted LIVE split: **3297 / 500 / 1000**
- grouped 5-fold CV on the 3297 training rows only
- 500-row competition validation stays untouched until the final gate
- ensemble weight chosen from train OOF AP only
- all candidate submissions preserve the V38 sexual positive count
- every non-sexual label stays exactly V38
- fold-level checkpoint/resume for the transformer

At the end, upload `v44_results.zip` to ChatGPT before submitting anything to Kaggle.


In [ ]:

# 1. Upload V44 LIVE bundle
from google.colab import files
import pathlib, zipfile

uploaded = files.upload()
assert "v44_LIVE_sexual_specialist_data.zip" in uploaded

DATA_DIR = pathlib.Path("/content/v44_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v44_LIVE_sexual_specialist_data.zip") as z:
    z.extractall(DATA_DIR)

for p in sorted(DATA_DIR.iterdir()):
    print(p.name, f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Install + preflight
import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.6",
    "tokenizers==0.22.1",
    "sentencepiece==0.2.1",
    "scikit-learn>=1.4",
    "scipy",
    "safetensors",
    "tqdm",
])

from transformers import AutoConfig, AutoTokenizer

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base"

print("Checking:", MODEL_NAME)
_ = AutoConfig.from_pretrained(MODEL_NAME)
_ = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ V44 checkpoint accessible")


In [ ]:

# 3. Imports and configuration

import gc, re, math, json, random, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

SEED = 2026
N_FOLDS = 5
EPOCHS = 3
SNAPSHOT_EPOCHS = [2,3]

TARGET = "sexual_content"
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base"

MAX_LENGTH = 128
TRAIN_BATCH = 6
EVAL_BATCH = 32
ACCUM = 3

BASE_LR = 1.4e-5
HEAD_MULT = 3.0
LAYER_DECAY = 0.90
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

RANK_WEIGHT = 0.20
FGM_WEIGHT = 0.30
FGM_EPS = 0.50

LABELS = [
    "is_toxic",
    "implicit_toxicity",
    "sarcasm",
    "contains_profanity",
    "threatening",
    "identity_attack",
    "sexual_content",
]

USE_DRIVE_CHECKPOINTS = True

if USE_DRIVE_CHECKPOINTS:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/V44_Sexual_Checkpoints")
else:
    RUN_DIR = Path("/content/v44_output")

RUN_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR = Path("/content/v44_export")
EXPORT_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_all(SEED)

assert torch.cuda.is_available(), "Enable a T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint dir:", RUN_DIR)


In [ ]:

# 4. Load and verify exact LIVE partition

train = pd.read_csv(DATA_DIR/"train.csv", dtype={"ID":str})
valid = pd.read_csv(DATA_DIR/"validation.csv", dtype={"ID":str})
test = pd.read_csv(DATA_DIR/"test.csv", dtype={"ID":str})
v2v = pd.read_csv(DATA_DIR/"v2_valid.csv", dtype={"ID":str})
v2t = pd.read_csv(DATA_DIR/"v2_test.csv", dtype={"ID":str})
champ = pd.read_csv(DATA_DIR/"submission_v38_champion.csv", dtype={"ID":str})

assert len(train) == 3297
assert len(valid) == 500
assert len(test) == 1000

assert valid["ID"].tolist() == v2v["ID"].tolist()
assert test["ID"].tolist() == v2t["ID"].tolist()
assert test["ID"].tolist() == champ["ID"].tolist()

assert test["ID"].head(4).tolist() == [
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

def clean_text(x):
    x = unicodedata.normalize("NFKC", str(x))
    x = x.replace("\u200c"," ").replace("\u200d"," ")
    x = re.sub(r"https?://\S+|www\.\S+", " URL ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

for df in [train,valid,test]:
    df["clean_text"] = df["text"].map(clean_text)
    df["group_text"] = df["clean_text"]

print("✅ exact LIVE IDs verified")
print()
print("Sexual-content positives:")
print("train:", int(train[TARGET].sum()))
print("valid:", int(valid[TARGET].sum()))
print("V38 test predictions:", int(champ[TARGET].sum()))

hard_prof = (
    (train["contains_profanity"]==1)
    & (train[TARGET]==0)
).sum()

print("profane-but-not-sexual TRAIN hard negatives:", int(hard_prof))


In [ ]:

# 5. Grouped TRAIN-only folds

# Stratify on sexual label + profanity relation so each fold gets
# explicit sexual positives and profane hard negatives.
strata = (
    train[TARGET].astype(str)
    + "_"
    + train["contains_profanity"].astype(str)
)

sgkf = StratifiedGroupKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

train["fold"] = -1

for fold, (_, va_idx) in enumerate(
    sgkf.split(
        train,
        y=strata,
        groups=train["group_text"]
    )
):
    train.loc[va_idx,"fold"] = fold

assert (train["fold"] >= 0).all()
assert train.groupby("group_text")["fold"].nunique().max() == 1

display(
    train.groupby("fold").agg(
        rows=("ID","size"),
        sexual_pos=(TARGET,"sum"),
        profanity_pos=("contains_profanity","sum"),
    )
)


In [ ]:

# 6. NB-SVM lexical expert
#
# Honest fold-specific vectorizers are fitted for OOF predictions.
# Then a full-train model predicts external validation and test.

def nb_log_ratio(X, y):
    y = np.asarray(y, int)

    p = np.asarray(
        X[y==1].sum(axis=0)
    ).ravel() + 1.0

    q = np.asarray(
        X[y==0].sum(axis=0)
    ).ravel() + 1.0

    p /= p.sum()
    q /= q.sum()

    return np.log(p/q)


def make_vectorizers():
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1,2),
        min_df=2,
        max_features=60000,
        sublinear_tf=True,
        strip_accents=None,
    )

    char = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3,5),
        min_df=2,
        max_features=100000,
        sublinear_tf=True,
    )

    return word,char


def fit_nbsvm(train_text, y, pred_texts):
    word,char = make_vectorizers()

    Xw = word.fit_transform(train_text)
    Xc = char.fit_transform(train_text)
    X = hstack([Xw,Xc],format="csr")

    r = nb_log_ratio(X,y)
    Xr = X.multiply(r)

    # Mild positive weighting, not fully balanced.
    sw = np.where(
        np.asarray(y)==1,
        2.2,
        1.0
    )

    clf = LogisticRegression(
        C=2.0,
        solver="liblinear",
        max_iter=1500,
        random_state=SEED
    )

    clf.fit(
        Xr,
        y,
        sample_weight=sw
    )

    preds = []

    for texts in pred_texts:
        Pw = word.transform(texts)
        Pc = char.transform(texts)
        P = hstack([Pw,Pc],format="csr")
        preds.append(
            clf.predict_proba(
                P.multiply(r)
            )[:,1]
        )

    return preds


nb_oof = np.zeros(len(train),np.float32)

for fold in range(N_FOLDS):
    tr = train[train["fold"]!=fold]
    va = train[train["fold"]==fold]

    [pv] = fit_nbsvm(
        tr["clean_text"].tolist(),
        tr[TARGET].to_numpy(int),
        [va["clean_text"].tolist()]
    )

    nb_oof[va.index.to_numpy()] = pv
    print("NB-SVM fold",fold,"done")

# Full train -> untouched validation + test
nb_valid, nb_test = fit_nbsvm(
    train["clean_text"].tolist(),
    train[TARGET].to_numpy(int),
    [
        valid["clean_text"].tolist(),
        test["clean_text"].tolist()
    ]
)

print(
    "TRAIN OOF NB-SVM AP:",
    average_precision_score(
        train[TARGET],
        nb_oof
    )
)


In [ ]:

# 7. Transformer helpers

class SexualDataset(Dataset):
    def __init__(self, df, tokenizer, labels=True):
        self.text = df["clean_text"].astype(str).tolist()
        self.y = df[TARGET].to_numpy(np.float32) if labels else None
        self.tok = tokenizer

    def __len__(self):
        return len(self.text)

    def __getitem__(self,i):
        item = self.tok(
            self.text[i],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False,
        )

        if self.y is not None:
            item["labels"] = self.y[i]

        return item


class PadCollator:
    def __init__(self,tok):
        self.tok = tok

    def __call__(self,items):
        y = None

        if "labels" in items[0]:
            y = torch.tensor(
                [float(x.pop("labels")) for x in items],
                dtype=torch.float32
            )

        batch = self.tok.pad(
            items,
            padding=True,
            pad_to_multiple_of=8,
            return_tensors="pt"
        )

        if y is not None:
            batch["labels"] = y

        return batch


def pairwise_rank_loss(logits,y):
    pos = logits[y>.5]
    neg = logits[y<=.5]

    if len(pos)==0 or len(neg)==0:
        return logits.new_tensor(0.0)

    if len(pos)>16:
        pos = pos[:16]

    if len(neg)>32:
        neg = neg[:32]

    return torch.nn.functional.softplus(
        -(pos[:,None]-neg[None,:])
    ).mean()


class FGM:
    def __init__(self,model,epsilon):
        self.model = model
        self.epsilon = epsilon
        self.backup = {}

    def attack(self):
        for name,param in self.model.named_parameters():
            if (
                param.requires_grad
                and param.grad is not None
                and "word_embeddings" in name
            ):
                norm = torch.norm(param.grad)

                if torch.isfinite(norm) and norm>0:
                    self.backup[name] = param.data.clone()
                    param.data.add_(
                        self.epsilon*param.grad/norm
                    )

    def restore(self):
        for name,param in self.model.named_parameters():
            if name in self.backup:
                param.data = self.backup[name]

        self.backup = {}


def build_optimizer(model):
    named = list(model.named_parameters())
    rx = re.compile(r"encoder\.layer\.(\d+)")

    ids = []

    for name,_ in named:
        m = rx.search(name)
        if m:
            ids.append(int(m.group(1)))

    max_layer = max(ids) if ids else 11
    groups = {}

    for name,p in named:
        if not p.requires_grad:
            continue

        is_head = (
            "classifier" in name
            or name.startswith("score")
        )

        m = rx.search(name)

        if is_head:
            lr = BASE_LR*HEAD_MULT
        elif m:
            layer = int(m.group(1))
            lr = BASE_LR*(LAYER_DECAY**(max_layer-layer))
        else:
            lr = BASE_LR*(LAYER_DECAY**(max_layer+1))

        no_decay = (
            name.endswith("bias")
            or "LayerNorm.weight" in name
            or "layer_norm.weight" in name
        )

        wd = 0.0 if no_decay else WEIGHT_DECAY
        key = (round(lr,12),wd)

        groups.setdefault(key,[]).append(p)

    return torch.optim.AdamW([
        {"params":ps,"lr":lr,"weight_decay":wd}
        for (lr,wd),ps in groups.items()
    ])


@torch.no_grad()
def predict(model,loader):
    model.eval()
    out = []

    for batch in loader:
        batch.pop("labels",None)

        inputs = {
            k:v.to("cuda",non_blocking=True)
            for k,v in batch.items()
        }

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = model(**inputs).logits.squeeze(-1)

        out.append(
            torch.sigmoid(
                logits.float()
            ).cpu().numpy()
        )

    return np.concatenate(out)


In [ ]:

# 8. Train 5-fold Twitter-XLM-R sexual specialist

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
collator = PadCollator(tokenizer)

xlmr_oof = np.zeros(len(train),np.float32)
valid_parts = []
test_parts = []
history = []

fold_seeds = [2026,17,3407,1337,42]

for fold in range(N_FOLDS):
    tr = train[train["fold"]!=fold].copy()
    va = train[train["fold"]==fold].copy()

    cache = RUN_DIR/f"xlmr_fold_{fold}.npz"
    hp = RUN_DIR/f"xlmr_fold_{fold}_history.csv"

    if cache.exists():
        try:
            with np.load(cache) as z:
                idx = z["idx"]
                poof = z["oof"]
                pvalid = z["external_valid"]
                ptest = z["test"]

            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse XLM-R fold",fold)
                xlmr_oof[idx] = poof
                valid_parts.append(pvalid)
                test_parts.append(ptest)

                if hp.exists():
                    history.extend(
                        pd.read_csv(hp).to_dict("records")
                    )

                continue
        except Exception as e:
            print("Ignoring bad cache:",e)

    seed_all(fold_seeds[fold])

    sexual = tr[TARGET].to_numpy(int)
    profanity = tr["contains_profanity"].to_numpy(int)
    toxic = tr["is_toxic"].to_numpy(int)

    # Positive emphasis + explicit hard negatives.
    sample_w = np.ones(len(tr),float)

    sample_w[sexual==1] = 3.2

    hard_prof_neg = (
        (sexual==0)
        & (profanity==1)
    )
    sample_w[hard_prof_neg] = 2.0

    hard_toxic_neg = (
        (sexual==0)
        & (toxic==1)
        & (~hard_prof_neg)
    )
    sample_w[hard_toxic_neg] = 0.90

    sample_w[
        (sexual==0)
        & (toxic==0)
    ] = 0.45

    sampler = WeightedRandomSampler(
        torch.tensor(
            sample_w,
            dtype=torch.double
        ),
        num_samples=len(tr),
        replacement=True
    )

    tr_loader = DataLoader(
        SexualDataset(tr,tokenizer,True),
        batch_size=TRAIN_BATCH,
        sampler=sampler,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator
    )

    va_loader = DataLoader(
        SexualDataset(va,tokenizer,True),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator
    )

    ext_loader = DataLoader(
        SexualDataset(valid,tokenizer,True),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator
    )

    test_loader = DataLoader(
        SexualDataset(test,tokenizer,False),
        batch_size=EVAL_BATCH,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
        collate_fn=collator
    )

    gc.collect()
    torch.cuda.empty_cache()

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=1
    ).to("cuda")

    if hasattr(model,"gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()

    optimizer = build_optimizer(model)

    total_steps = math.ceil(
        len(tr_loader)/ACCUM
    )*EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        int(total_steps*WARMUP_RATIO),
        total_steps
    )

    scaler = torch.amp.GradScaler("cuda")
    fgm = FGM(model,FGM_EPS)

    oof_snaps = []
    ext_snaps = []
    test_snaps = []
    fold_hist = []

    for epoch in range(1,EPOCHS+1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running = 0.0
        seen = 0

        for step,batch in enumerate(
            tqdm(
                tr_loader,
                desc=f"XLM-R f{fold} e{epoch}/{EPOCHS}"
            )
        ):
            y = batch.pop("labels").to(
                "cuda",
                non_blocking=True
            )

            inputs = {
                k:v.to("cuda",non_blocking=True)
                for k,v in batch.items()
            }

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(**inputs).logits.squeeze(-1)

            # Mild main-task positive boost
            base = torch.nn.functional.binary_cross_entropy_with_logits(
                logits.float(),
                y,
                reduction="none"
            )

            class_w = torch.where(
                y>.5,
                torch.full_like(y,1.55),
                torch.ones_like(y)
            )

            bce = (base*class_w).mean()
            rank = pairwise_rank_loss(logits.float(),y)

            loss = bce + RANK_WEIGHT*rank

            if not torch.isfinite(loss):
                raise FloatingPointError("non-finite V44 loss")

            scaler.scale(
                loss/ACCUM
            ).backward()

            if epoch >= 2:
                fgm.attack()

                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16
                ):
                    adv_logits = model(
                        **inputs
                    ).logits.squeeze(-1)

                adv_base = torch.nn.functional.binary_cross_entropy_with_logits(
                    adv_logits.float(),
                    y,
                    reduction="none"
                )

                adv_bce = (
                    adv_base*class_w
                ).mean()

                adv_rank = pairwise_rank_loss(
                    adv_logits.float(),
                    y
                )

                adv_loss = (
                    adv_bce
                    +RANK_WEIGHT*adv_rank
                )

                scaler.scale(
                    FGM_WEIGHT*adv_loss/ACCUM
                ).backward()

                fgm.restore()

            running += float(loss.item())*len(y)
            seen += len(y)

            if (
                (step+1)%ACCUM==0
                or step+1==len(tr_loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0
                )

                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()

                if scaler.get_scale() >= old_scale:
                    scheduler.step()

                optimizer.zero_grad(set_to_none=True)

        rec = {
            "fold":fold,
            "epoch":epoch,
            "loss":running/max(seen,1)
        }

        fold_hist.append(rec)
        history.append(rec)

        pd.DataFrame(fold_hist).to_csv(
            hp,index=False
        )

        print(rec)

        if epoch in SNAPSHOT_EPOCHS:
            oof_snaps.append(
                predict(model,va_loader)
            )
            ext_snaps.append(
                predict(model,ext_loader)
            )
            test_snaps.append(
                predict(model,test_loader)
            )

    poof = np.mean(oof_snaps,axis=0)
    pvalid = np.mean(ext_snaps,axis=0)
    ptest = np.mean(test_snaps,axis=0)

    idx = va.index.to_numpy()

    xlmr_oof[idx] = poof
    valid_parts.append(pvalid)
    test_parts.append(ptest)

    np.savez_compressed(
        cache,
        idx=idx,
        oof=poof,
        external_valid=pvalid,
        test=ptest
    )

    del model,optimizer,scheduler,scaler
    del tr_loader,va_loader,ext_loader,test_loader
    gc.collect()
    torch.cuda.empty_cache()

xlmr_valid = np.mean(valid_parts,axis=0)
xlmr_test = np.mean(test_parts,axis=0)

np.savez_compressed(
    RUN_DIR/"xlmr_complete.npz",
    oof=xlmr_oof,
    external_valid=xlmr_valid,
    test=xlmr_test
)

pd.DataFrame(history).to_csv(
    RUN_DIR/"xlmr_history.csv",
    index=False
)

print(
    "TRAIN OOF XLM-R AP:",
    average_precision_score(
        train[TARGET],
        xlmr_oof
    )
)


In [ ]:

# 9. Choose NB-SVM / XLM-R ensemble using TRAIN OOF only

def logit(p):
    p = np.clip(
        np.asarray(p,float),
        1e-6,1-1e-6
    )
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

y_train = train[TARGET].to_numpy(int)

recipes = []

for w_xlmr in [0.0,0.20,0.40,0.60,0.80,1.0]:
    score = sigmoid(
        w_xlmr*logit(xlmr_oof)
        +(1-w_xlmr)*logit(nb_oof)
    )

    ap = average_precision_score(
        y_train,
        score
    )

    recipes.append(
        (ap,w_xlmr)
    )

recipes.sort(reverse=True)

best_oof_ap,w_xlmr = recipes[0]

print("TRAIN-only selected XLM-R weight:",w_xlmr)
print("TRAIN OOF ensemble AP:",best_oof_ap)

new_valid_score = sigmoid(
    w_xlmr*logit(xlmr_valid)
    +(1-w_xlmr)*logit(nb_valid)
)

new_test_score = sigmoid(
    w_xlmr*logit(xlmr_test)
    +(1-w_xlmr)*logit(nb_test)
)


In [ ]:

# 10. Candidate construction — fixed V38 counts

def f1_pos(y,p):
    y = np.asarray(y,int)
    p = np.asarray(p,int)

    tp = ((y==1)&(p==1)).sum()
    fp = ((y==0)&(p==1)).sum()
    fn = ((y==1)&(p==0)).sum()

    d = 2*tp+fp+fn
    return float(2*tp/d) if d else 0.0


def topk(prob,k):
    out = np.zeros(len(prob),int)
    k = int(max(0,min(k,len(prob))))

    if k:
        out[np.argsort(-prob)[:k]] = 1

    return out


BASE_THRESHOLD = 0.245

base_valid_score = v2v[
    "sexual_content_prob"
].to_numpy(float)

base_test_score = v2t[
    "sexual_content_prob"
].to_numpy(float)

base_valid_pred = (
    base_valid_score >= BASE_THRESHOLD
).astype(int)

base_test_pred = champ[TARGET].to_numpy(int)

k_valid = int(base_valid_pred.sum())
k_test = int(base_test_pred.sum())

# Predetermined conservative blends.
candidate_scores = {}

for new_weight in [0.25,0.50,0.75,1.00]:
    sv = sigmoid(
        (1-new_weight)*logit(base_valid_score)
        +new_weight*logit(new_valid_score)
    )

    st = sigmoid(
        (1-new_weight)*logit(base_test_score)
        +new_weight*logit(new_test_score)
    )

    candidate_scores[
        f"new{int(new_weight*100):02d}_fixed_count"
    ] = (
        sv,
        st,
        topk(sv,k_valid),
        topk(st,k_test)
    )


In [ ]:

# 11. Untouched 500-row validation gate
#
# First use of validation sexual-content labels.

y_valid = valid[TARGET].to_numpy(int)

baseline_f1 = f1_pos(
    y_valid,
    base_valid_pred
)

baseline_ap = average_precision_score(
    y_valid,
    base_valid_score
)

rows = []

for name,(sv,st,pv,pt) in candidate_scores.items():
    f1 = f1_pos(y_valid,pv)
    ap = average_precision_score(y_valid,sv)

    rows.append({
        "candidate":name,
        "validation_f1":f1,
        "validation_gain":f1-baseline_f1,
        "validation_ap":ap,
        "ap_gain":ap-baseline_ap,
        "validation_positive_count":int(pv.sum()),
        "test_positive_count":int(pt.sum()),
        "test_changes_vs_v38":int(
            (pt!=base_test_pred).sum()
        ),
    })

report = pd.DataFrame(rows).sort_values(
    ["validation_f1","validation_ap"],
    ascending=False
).reset_index(drop=True)

print("V38 sexual validation F1:",baseline_f1)
print("V38 sexual validation AP:",baseline_ap)
display(report.round(6))

best = report.iloc[0].to_dict()

# Sexual validation has only 24 positives, so require a large gain
# in both F1 and ranking quality before risking a public submission.
PASS = bool(
    best["validation_gain"] >= 0.055
    and best["ap_gain"] >= 0.035
    and best["test_changes_vs_v38"] <= 100
)

summary = {
    "protected_public_lb":0.608,
    "target_public_lb":0.620,
    "baseline_sexual_f1":float(baseline_f1),
    "baseline_sexual_ap":float(baseline_ap),
    "train_oof_selected_xlmr_weight":float(w_xlmr),
    "train_oof_ensemble_ap":float(best_oof_ap),
    "best_candidate":best,
    "PASS":PASS,
}

print(json.dumps(summary,indent=2))


In [ ]:

# 12. Create V44 candidate submissions
#
# Only sexual_content may change.

submission_paths = {}

for name,(_,_,_,pt) in candidate_scores.items():
    sub = champ.copy()
    sub[TARGET] = pt.astype(int)

    assert sub["ID"].tolist() == test["ID"].tolist()
    assert sub["ID"].head(4).tolist() == [
        "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
    ]

    for label in LABELS:
        if label != TARGET:
            assert np.array_equal(
                sub[label].to_numpy(int),
                champ[label].to_numpy(int)
            )

    assert sub[LABELS].isin([0,1]).all().all()

    path = EXPORT_DIR/f"submission_v44_{name}.csv"
    sub.to_csv(path,index=False)
    submission_paths[name] = str(path)

recommended_name = report.iloc[0]["candidate"]

if PASS:
    rec = pd.read_csv(
        submission_paths[recommended_name],
        dtype={"ID":str}
    )

    rec_path = EXPORT_DIR/"submission_v44_RECOMMENDED.csv"
    rec.to_csv(rec_path,index=False)

    print("✅ V44 PASSED")
    print("Recommended:",recommended_name)
else:
    print("❌ V44 did not pass.")
    print("Do not submit. Upload v44_results.zip to ChatGPT.")


In [ ]:

# 13. Package results

np.savez_compressed(
    EXPORT_DIR/"v44_sexual_probabilities.npz",
    nb_oof=nb_oof,
    nb_valid=nb_valid,
    nb_test=nb_test,
    xlmr_oof=xlmr_oof,
    xlmr_valid=xlmr_valid,
    xlmr_test=xlmr_test,
    new_valid_score=new_valid_score,
    new_test_score=new_test_score,
)

report.to_csv(
    EXPORT_DIR/"v44_validation_report.csv",
    index=False
)

with open(
    EXPORT_DIR/"v44_summary.json",
    "w"
) as f:
    json.dump(summary,f,indent=2)

zip_path = Path("/content/v44_results.zip")

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:
    for p in EXPORT_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
